# Setup


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Recall @ k curves, baseline and ours


In [ ]:
HOPS_TO_SPLIT = {
    "all": "All",
    "1-hop": "1-hop",
    "2-hop": "Multi-hop",
}


def load_recall_csv(*paths: str) -> pd.DataFrame:
    frames = [pd.read_csv(p) for p in paths]
    df = pd.concat(frames, ignore_index=True)

    df = df[df["hops"].isin(HOPS_TO_SPLIT)]
    df["Split"] = df["hops"].map(HOPS_TO_SPLIT)

    df = df.dropna(subset=["value"])

    df = df.rename(columns={"model": "Model"})
    return df


def plot_triple_recall_at_k(
    df: pd.DataFrame,
    save_path: str | None = None,
    figsize: tuple[float, float] = (7.5, 4.0),
):

    sub = df[df["metric"] == "triple_recall"].copy()
    k_values = sorted(sub["k"].unique())

    sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)
    palette = sns.color_palette("colorblind")

    splits = ["All", "Multi-hop"]
    fig, axes = plt.subplots(1, 2, figsize=(figsize[0], figsize[1]))

    for ax, split in zip(axes, splits):
        chunk = sub[sub["Split"] == split]
        if chunk.empty:
            continue
        sns.lineplot(
            data=chunk,
            x="k",
            y="value",
            hue="Model",
            style="Model",
            markers=True,
            dashes=False,
            markersize=8,
            linewidth=2,
            palette=palette,
            ax=ax,
        )
        ax.set_title(split, fontweight="bold")
        ax.set_xlabel("$k$")
        ax.set_xticks(k_values)
        ax.set_xticklabels(k_values)

        ymin_raw = chunk["value"].min()
        ymin = np.floor(ymin_raw / 0.05) * 0.05
        ax.set_ylim(ymin, 1.025)
        ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))

        ax.set_ylabel("Triple Recall@$k$")
        if ax != axes[0]:
            ax.get_legend().remove()
        else:
            ax.legend(frameon=True, loc="lower right", fontsize=10)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"Saved to {save_path}")

    return fig


df = load_recall_csv(
    "../data/baseline/recall_by_hops.csv",
    "../data/ours/recall_by_hops.csv",
)
plot_triple_recall_at_k(df, save_path="./figures/triple_recall_at_k.pdf")

# Recall @ k curves (using no-gating model from ablations.json)


In [ ]:
import json

# Load no-gating results from ablations.json as "ours"
with open("../data/ours/ablations.json") as f:
    ablations_raw = json.load(f)

no_gating = ablations_raw["no-gating"]

# Reshape ablations.json flat dict into the same long-form DataFrame
# Keys are like: triple_recall@50, triple_recall@50_1hop, triple_recall@50_multihop
SPLIT_MAP = {
    "": ("all", "All"),
    "_1hop": ("1-hop", "1-hop"),
    "_multihop": ("2-hop", "Multi-hop"),
}

ours_rows = []
for metric in ["triple_recall", "entity_recall"]:
    for k in [50, 100, 200, 400]:
        for suffix, (hops_key, split_label) in SPLIT_MAP.items():
            key = f"{metric}@{k}{suffix}"
            val = no_gating.get(key)
            if val is not None:
                csv_metric = "ans_recall" if metric == "entity_recall" else metric
                ours_rows.append(
                    {
                        "Model": "Ours",
                        "hops": hops_key,
                        "Split": split_label,
                        "metric": csv_metric,
                        "k": k,
                        "value": val,
                    }
                )

ours_df = pd.DataFrame(ours_rows)

# Load baseline from CSV
baseline_df = load_recall_csv("../data/baseline/recall_by_hops.csv")

# Combine
combined = pd.concat([baseline_df, ours_df], ignore_index=True)

plot_triple_recall_at_k(combined, save_path="./figures/triple_recall_at_k_nogating.pdf")

# Multi-seed averaged test metrics for Ours (gated) and w/o Gating


In [ ]:
def load_averaged_metrics(csv_path: str) -> dict[str, float]:
    """Load recall_by_hops_averaged.csv and return a flat dict keyed like triple_recall@50."""
    df = pd.read_csv(csv_path)
    out = {}
    for _, row in df.iterrows():
        if row["hops"] != "all":
            continue
        metric_key = "entity_recall" if row["metric"] == "ans_recall" else row["metric"]
        out[f"{metric_key}@{int(row['k'])}"] = row["mean"]
        out[f"{metric_key}@{int(row['k'])}_std"] = row["std"]
    return out


ours_avg = load_averaged_metrics(
    "../data/ours/seed_evals/gated/recall_by_hops_averaged.csv"
)
nogating_avg = load_averaged_metrics(
    "../data/ours/seed_evals/recall_by_hops_averaged.csv"
)

print("Ours (gated, 3-seed avg):")
for k, v in sorted(ours_avg.items()):
    if not k.endswith("_std"):
        std = ours_avg.get(f"{k}_std", 0)
        print(f"  {k}: {v:.4f} +/- {std:.4f}")

print("\nw/o Gating (3-seed avg):")
for k, v in sorted(nogating_avg.items()):
    if not k.endswith("_std"):
        std = nogating_avg.get(f"{k}_std", 0)
        print(f"  {k}: {v:.4f} +/- {std:.4f}")

# Ablation study table


In [ ]:
import json

with open("../data/ours/ablations.json") as f:
    ablations_raw = json.load(f)


def extract_all_metrics(csv_path: str) -> dict[str, float]:
    df = pd.read_csv(csv_path)
    out = {}
    for _, row in df.iterrows():
        if row["hops"] == "all" and row["metric"] == "triple_recall":
            out[f"triple_recall@{int(row['k'])}"] = row["value"]
        elif row["hops"] == "all" and row["metric"] == "ans_recall":
            out[f"entity_recall@{int(row['k'])}"] = row["value"]
    return out


baseline_metrics = extract_all_metrics("../data/baseline/recall_by_hops.csv")

DISPLAY_COLS = [
    ("triple_recall", 50),
    ("triple_recall", 100),
    ("triple_recall", 200),
    ("entity_recall", 100),
]

ROWS = [
    ("baseline", "SubgraphRAG", baseline_metrics),
    ("ours", "Ours", ours_avg),
    ("no-topic-pe", "w/o Topic PE", ablations_raw["no-topic-pe"]),
    ("no-anchor-pe", "w/o Anchor PE", ablations_raw["no-anchor-pe"]),
    ("no-path-pe", "w/o Path PE", ablations_raw["no-path-pe"]),
    ("no-pe", "w/o All PE", ablations_raw["no-pe"]),
    (
        "no-gating",
        "w/o Gating",
        nogating_avg,
    ),  # averaged over 3 no-gating seeds from W&B
    ("no-gnn", "w/o GNN", ablations_raw["no-gnn"]),
]

col_keys = [f"{m}@{k}" for m, k in DISPLAY_COLS]

rows = []
for key, label, src in ROWS:
    row = {"Variant": label}
    for col_key in col_keys:
        row[col_key] = src.get(col_key)
    rows.append(row)

table = pd.DataFrame(rows).set_index("Variant")

table = table.round(3)

ours_row = table.loc["Ours"]
delta = table.subtract(ours_row)


def format_percent(v):
    if pd.isna(v):
        return "-"
    return f"{v * 100:.1f}"


def format_delta(v):
    if pd.isna(v) or v == 0:
        return ""
    return f" ({v * 100:+.1f})"


def build_display(bold_fn):
    display_rows = []
    for variant in table.index:
        row = {"Variant": variant}
        for col in table.columns:
            val = format_percent(table.loc[variant, col])
            if variant == "Ours":
                row[col] = bold_fn(val)
            else:
                d = format_delta(delta.loc[variant, col])
                row[col] = f"{val}{d}"
        display_rows.append(row)
    df = pd.DataFrame(display_rows).set_index("Variant")
    col_rename = {}
    for col in df.columns:
        metric, k = col.rsplit("@", 1)
        nice = "TR" if metric == "triple_recall" else "ER"
        col_rename[col] = f"{nice}@{k}"
    return df.rename(columns=col_rename)


latex_df = build_display(lambda v: rf"\textbf{{{v}}}")
latex_df.index = [rf"\textbf{{{v}}}" if v == "Ours" else v for v in latex_df.index]
latex_df.to_latex(
    "figures/ablations.tex",
    escape=False,
    column_format="l" + "c" * len(DISPLAY_COLS),
)
print("Saved figures/ablations.tex")